In [3]:
!pip install interpret

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 11.8 MB/s eta 0:00:0000:010:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.3/46.3 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 48.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 60.0 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 64.0 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 58.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.1/780.1 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 61.3 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 43.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 19.5 MB/s eta 0:00:00
  Created wheel for dash

In [ ]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.preprocessing import StandardScaler
from interpret.glassbox import ExplainableBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

# =====================================================
# 1. CONFIGURATION (CHANGE ONLY THIS PART)
# =====================================================

DATA_PATH = r"C:\Users\Sam\Desktop\ML\task\Data.xlsx"
sheet_name = "Data_after_KFold_EBM"
CONFIG = {
    "optimizer": "HEOA",
    "population": 25,
    "iterations": 200,
    "cv": 5,
    "random_state": 42
}

# =====================================================
# 2. LOAD DATA
# =====================================================

df = pd.read_excel(DATA_PATH, sheet_name=sheet_name)
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

X = StandardScaler().fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=CONFIG["random_state"]
)

# Bounds specific to Explainable Boosting Regressor
MODEL = {
    "name": "ExplainableBoostingRegressor",
    "builder": ExplainableBoostingRegressor,
    "bounds": {
        "learning_rate": (0.001, 0.2, float),
        "max_leaves": (2, 50, int),
        "max_bins": (32, 256, int)
    }
}

# =====================================================
# 3. HELPER FUNCTIONS
# =====================================================

def bounds_to_arrays(bounds):
    lb, ub, cast = [],[], []
    for v in bounds.values():
        lb.append(v[0])
        ub.append(v[1])
        cast.append(v[2])
    return np.array(lb), np.array(ub), cast

def decode_params(vec, bounds, cast):
    decoded = {}
    for i, k in enumerate(bounds.keys()):
        val = cast[i](vec[i])
        decoded[k] = val
    return decoded

def make_objective(model_builder, bounds, cast):
    def objective(vec):
        params = decode_params(vec, bounds, cast)
        
        # Build a much faster EBM
        model = model_builder(
            **params,
            interactions=0,           # No interactions
            outer_bags=1,             # 🔥 HUGE SPEEDUP: Train 1 model instead of 8
            inner_bags=0,             # Disable inner bagging
            early_stopping_rounds=20, # 🔥 Stop early if it stops learning
            random_state=CONFIG["random_state"],
            n_jobs=-1                 # Let EBM use your CPU cores
        )

        scores = cross_validate(
            model,
            X_train,
            y_train,
            cv=CONFIG["cv"],
            scoring={"mse": "neg_mean_squared_error", "r2": "r2"},
            n_jobs=1                  # MUST STAY 1 to prevent Windows freeze
        )

        mse = -scores["test_mse"].mean()
        r2 = scores["test_r2"].mean()

        return mse, r2 
    return objective
# =====================================================
# 4. HUMAN EVOLUTIONARY OPTIMIZATION ALGORITHM (HEOA)
# =====================================================

def HEOA(objective, lb, ub, N, T, cast):
    start = time.time()
    D = len(lb)

    # --- Initialize population using Logistic Chaos Mapping ---
    pop_chaos = np.random.rand(N, D)
    for _ in range(50):  
        pop_chaos = 4.0 * pop_chaos * (1.0 - pop_chaos)
    pop = lb + pop_chaos * (ub - lb)
    
    # Initialize arrays to track fitness (MSE) and R2
    fit = np.zeros(N)
    fit_r2 = np.zeros(N)
    for i in range(N):
        print(f" -> Initializing individual {i+1}/{N}...")
        fit[i], fit_r2[i] = objective(pop[i])

    best_idx = np.argmin(fit)
    best = pop[best_idx].copy()
    best_fit = fit[best_idx]
    best_r2 = fit_r2[best_idx]

    convergence_mse = []
    convergence_r2 =[]
    log =[]

    for t in range(T):
        alpha = 1 - t / T
        mean_pop = np.mean(pop, axis=0)
        
        exploration_phase = (t < 0.2 * T)
        
        if exploration_phase:
            # Human Exploration Phase
            for i in range(N):
                r1 = np.random.rand(D)
                r2_val = np.random.randn(D) # renamed to r2_val to avoid confusion with R2 score
                candidate = pop[i] + r1 * (best - pop[i]) + alpha * r2_val * (mean_pop - pop[i])
                candidate = np.clip(candidate, lb, ub)
                
                f_mse, f_r2 = objective(candidate)

                if f_mse < fit[i]:
                    pop[i] = candidate
                    fit[i] = f_mse
                    fit_r2[i] = f_r2
                    if f_mse < best_fit:
                        best, best_fit, best_r2 = candidate.copy(), f_mse, f_r2
        else:
            # Human Development Phase
            sorted_idx = np.argsort(fit)
            n_leaders = max(1, int(0.1 * N))
            n_explorers = max(1, int(0.3 * N))
            n_followers = max(1, int(0.4 * N))
            
            for rank, i in enumerate(sorted_idx):
                if rank < n_leaders:
                    candidate = best + alpha * np.random.randn(D) * (best - pop[i])
                elif rank < n_leaders + n_explorers:
                    leader_idx = sorted_idx[np.random.randint(0, n_leaders)]
                    candidate = pop[i] + np.random.rand(D) * (pop[leader_idx] - pop[i])
                elif rank < n_leaders + n_explorers + n_followers:
                    candidate = pop[i] + np.random.rand(D) * (mean_pop - pop[i])
                else:
                    candidate = lb + np.random.rand(D) * (ub - lb)

                candidate = np.clip(candidate, lb, ub)
                f_mse, f_r2 = objective(candidate)

                if f_mse < fit[i]:
                    pop[i] = candidate
                    fit[i] = f_mse
                    fit_r2[i] = f_r2
                    if f_mse < best_fit:
                        best, best_fit, best_r2 = candidate.copy(), f_mse, f_r2

        # --- Record convergence per iteration ---
        convergence_mse.append(best_fit)
        convergence_r2.append(best_r2)

        # --- Log the best solution at this iteration ---
        best_decoded = decode_params(best, MODEL["bounds"], cast)
        log.append([t + 1] + [best_decoded[k] for k in MODEL["bounds"].keys()] +[best_fit, best_r2])

        # --- Print real-time updates ---
        print(
            f"Iter {t+1:03d}/{T} | "
            + ", ".join(f"{k}={v}" for k, v in best_decoded.items())
            + f" | MSE: {best_fit:.4f} | R2: {best_r2:.4f}"
        )

    runtime = time.time() - start
    return decode_params(best, MODEL["bounds"], cast), best_fit, convergence_mse, convergence_r2, runtime, log

# =====================================================
# 5. RUN OPTIMIZATION
# =====================================================

lb, ub, cast = bounds_to_arrays(MODEL["bounds"])
objective = make_objective(MODEL["builder"], MODEL["bounds"], cast)

print("Starting HEOA Optimization...")
best, best_score, convergence_mse, convergence_r2, runtime, log = HEOA(
    objective, lb, ub, CONFIG["population"], CONFIG["iterations"], cast
)

# =====================================================
# 6. FINAL MODEL & REPORT
# =====================================================

best_params = best
final_model = MODEL["builder"](**best_params, random_state=CONFIG["random_state"])
final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)
test_mse = mean_squared_error(y_test, y_pred)
test_r2 = r2_score(y_test, y_pred)

# =====================================================
# 7. TABLES (EXCEL / PAPER READY)
# =====================================================

# Full iterations log (Added R2)
iter_cols = ["iteration"] + list(MODEL["bounds"].keys()) + ["best_cv_mse", "best_cv_r2"]
iterations_df = pd.DataFrame(log, columns=iter_cols)

# Convergence per iteration
convergence_df = pd.DataFrame({
    "best_mse": convergence_mse,
    "best_r2": convergence_r2
})

# Summary
summary_df = pd.DataFrame([{
    "Model": MODEL["name"],
    "Optimizer": CONFIG["optimizer"],
    "Runtime_sec": round(runtime, 2),
    "Best_CV_MSE": best_score,
    "Test_MSE": test_mse,
    "Test_R2_Score": test_r2
}])

# Best hyperparameters table
best_params_df = pd.DataFrame({
    "parameters": list(best_params.keys()),
    "values": list(best_params.values())
})

# =========================
# PRINT RESULTS
# =========================
print("\n✅ Optimization Completed!")
print("========================================")
print("\n✅ Best Hyperparameters Table:")
print(best_params_df)

print("\n✅ Summary Table:")
print(summary_df)

print("\n✅ Iterations Log Preview (First 5 and Last 5):")
print(pd.concat([iterations_df.head(5), iterations_df.tail(5)])) 

print("\n✅ Convergence Preview (First 5 and Last 5):")
print(pd.concat([convergence_df.head(5), convergence_df.tail(5)]))

Starting HEOA Optimization...
 -> Initializing individual 1/25...
 -> Initializing individual 2/25...
 -> Initializing individual 3/25...
 -> Initializing individual 4/25...
 -> Initializing individual 5/25...
 -> Initializing individual 6/25...
 -> Initializing individual 7/25...
 -> Initializing individual 8/25...
 -> Initializing individual 9/25...
 -> Initializing individual 10/25...
 -> Initializing individual 11/25...
 -> Initializing individual 12/25...
 -> Initializing individual 13/25...
 -> Initializing individual 14/25...
 -> Initializing individual 15/25...
 -> Initializing individual 16/25...
 -> Initializing individual 17/25...
 -> Initializing individual 18/25...
 -> Initializing individual 19/25...
 -> Initializing individual 20/25...
 -> Initializing individual 21/25...
 -> Initializing individual 22/25...
 -> Initializing individual 23/25...
 -> Initializing individual 24/25...
 -> Initializing individual 25/25...
Iter 001/200 | learning_rate=0.044511860661952854, max

KeyboardInterrupt: 